# ODMR Analysis Notebook (Switching)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.size'] = 20
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['axes.linewidth'] = 1.5
import os
import yaml
from glob import glob
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.axes_grid1 import make_axes_locatable
from skimage.measure import block_reduce as skim_blockred
import spectralFitFunctions as sff
from multiprocessing import Pool
from ipywidgets import interact, Dropdown, IntSlider
from bokeh.io import push_notebook, show, output_notebook
from bokeh.plotting import figure
from bokeh.layouts import row, column
from bokeh.models import Range1d
output_notebook()
from scipy.stats import median_abs_deviation
from scipy.io import savemat

## Load data, average, normalize

The data comes packaged in 32-bit arrays saved in uncompressed binary format (`\*.npy`). `sigstack` refers to a stack of images with the MW driving ON and `refstack` refers to a stack of images with MW driving OFF. The first two axes (row, col) in any array are the image dimensions and the third axis (depth) represents the MW sweep frequency. 

In [ ]:
# DO NOT MODIFY THIS CELL!

# Load yaml file and display
with open("parameters.yaml", "r") as yamlfile:
    par = yaml.load(yamlfile, Loader=yaml.FullLoader)
[print(f'{key:<48}{value}') for key,value in par.items()];

# Parse dictionary and assign values to Python variables
numblock = par['Number of blocks']
centerfreq = par['Center frequency (MHz)']
span = par['Span (MHz)']
numfreq = par['Number of frequency points per block']
numsweep = par['Total number of sweeps']
saveperiod = par['Save period (sweeps)']
imwidth = par['Image width (px)']
imheight = par['Image height (px)']
xoffset = par['Image x-offset (px)']
yoffset = par['Image y-offset (px)']
switching = par['Current Switching Enabled']
swpcount1 = par['Number of sweeps completed, State 1']
swpcount2 = par['Number of sweeps completed, State 2']
print('\n')

In [ ]:
def load_data(switching):
    # Use the current working directory as the folder path
    folder_path = os.getcwd()

    # Find all .npy files
    files = glob(os.path.join(folder_path, "*.npy"))

    # Categorize files based on their naming pattern
    if switching:
        categorized_files = {'state_1': {'sigstack': {}, 'refstack': {}},'state_2': {'sigstack': {}, 'refstack': {}}}
    else:
        categorized_files = {'regular': {'sigstack': {}, 'refstack': {}}}

    for file in files:
        filename = os.path.basename(file)
        if switching:
            if filename.startswith("state_1_"):
                base_name = filename.split("state_1_")[1]
                if base_name.startswith("sigstack_"):
                    sweep_number = int(base_name.split('_')[1].replace('.npy', ''))
                    categorized_files['state_1']['sigstack'][sweep_number] = file
                elif base_name.startswith("refstack_"):
                    sweep_number = int(base_name.split('_')[1].replace('.npy', ''))
                    categorized_files['state_1']['refstack'][sweep_number] = file
            elif filename.startswith("state_2_"):
                base_name = filename.split("state_2_")[1]
                if base_name.startswith("sigstack_"):
                    sweep_number = int(base_name.split('_')[1].replace('.npy', ''))
                    categorized_files['state_2']['sigstack'][sweep_number] = file
                elif base_name.startswith("refstack_"):
                    sweep_number = int(base_name.split('_')[1].replace('.npy', ''))
                    categorized_files['state_2']['refstack'][sweep_number] = file
        else:
            if filename.startswith("sigstack_"):
                sweep_number = int(filename.split('_')[1].replace('.npy', ''))
                categorized_files['regular']['sigstack'][sweep_number] = file
            elif filename.startswith("refstack_"):
                sweep_number = int(filename.split('_')[1].replace('.npy', ''))
                categorized_files['regular']['refstack'][sweep_number] = file

    def find_max_sweep_pair(sigstack_dict, refstack_dict):
        # Find the common sweep numbers between sigstack and refstack
        common_sweeps = set(sigstack_dict.keys()) & set(refstack_dict.keys())
        if common_sweeps:
            # Find the maximum sweep number in the common sweeps
            max_sweep = max(common_sweeps)
            return max_sweep
        return None

    if not switching:
        max_sweep = find_max_sweep_pair(categorized_files['regular']['sigstack'],categorized_files['regular']['refstack'])
        sigstack_regular, refstack_regular = None, None
        fname=categorized_files['regular']['sigstack'][max_sweep]; sigstack_regular = np.load(fname); print('Loaded ' + fname)
        fname=categorized_files['regular']['refstack'][max_sweep]; refstack_regular = np.load(fname); print('Loaded ' + fname)
        out = {'regular': {'sigstack': sigstack_regular, 'refstack': refstack_regular}}
    else:
        max_sweep_state_1 = find_max_sweep_pair(categorized_files['state_1']['sigstack'],categorized_files['state_1']['refstack'])
        max_sweep_state_2 = find_max_sweep_pair(categorized_files['state_2']['sigstack'],categorized_files['state_2']['refstack'])
        sigstack_state_1_data, refstack_state_1_data = None, None
        sigstack_state_2_data, refstack_state_2_data = None, None
        idx = max_sweep_state_1 if max_sweep_state_1 < max_sweep_state_2 else max_sweep_state_2
        fname=categorized_files['state_1']['sigstack'][idx]; sigstack_state_1_data = np.load(fname); print('Loaded ' + fname)
        fname=categorized_files['state_1']['refstack'][idx]; refstack_state_1_data = np.load(fname); print('Loaded ' + fname)
        fname=categorized_files['state_2']['sigstack'][idx]; sigstack_state_2_data = np.load(fname); print('Loaded ' + fname)
        fname=categorized_files['state_2']['refstack'][idx]; refstack_state_2_data = np.load(fname); print('Loaded ' + fname)
        out = {'state_1': {'sigstack': sigstack_state_1_data, 'refstack': refstack_state_1_data},
               'state_2': {'sigstack': sigstack_state_2_data, 'refstack': refstack_state_2_data}}

    return out

# Example usage:
data = load_data(switching)

# Accessing the loaded data
if not switching:
    sigstack_regular = data['regular']['sigstack']
    refstack_regular = data['regular']['refstack']
    cdata = sigstack_regular/refstack_regular
    print('Normalization complete.\n')
else:
    sigstack_state_1 = data['state_1']['sigstack']
    refstack_state_1 = data['state_1']['refstack']
    sigstack_state_2 = data['state_2']['sigstack']
    refstack_state_2 = data['state_2']['refstack']
    cdata_s1 = sigstack_state_1/refstack_state_1
    cdata_s2 = sigstack_state_2/refstack_state_2
    print('Normalization complete.\n')

## Preview a raw image and display histogram as inset

In [ ]:
idx = 0 # previews the first frame, change as needed
fig, ax = plt.subplots(1, 1, figsize=[12, 10])
ins = inset_axes(ax, width=3, height=1.5, loc='upper right', borderpad=2)
arr = refstack_state_1[:,:,idx]/swpcount1
im = ax.imshow(arr,cmap='gray')
ax.set_xlabel('x [pixels]')
ax.set_ylabel('y [pixels]')
divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='5%', pad=.2)
cbar = fig.colorbar(im, cax=cax, orientation='vertical',format="%3.0f")
cbar.ax.tick_params(labelsize=16)
ins.hist(arr.ravel(),bins=np.arange(256),density=True,color='crimson')
ins.set_xlabel('Pixel value')
ins.tick_params(colors='crimson') 
ins.xaxis.label.set_color('crimson')
ins.set_facecolor((1,1,1,0))
ins.spines['left'].set_color((1,1,1,0))
ins.spines['right'].set_color((1,1,1,0))
ins.spines['top'].set_color((1,1,1,0))
ins.spines['bottom'].set_color('crimson')
ins.set_xticks([0,127,255])
ins.set_xlim(0,255)
ins.set_yticks([])
for item in ([ax.title, ax.xaxis.label, ax.yaxis.label] +
             ax.get_xticklabels() + ax.get_yticklabels()):
    item.set_fontsize(20)
for item in ([ins.title, ins.xaxis.label, ins.yaxis.label] +
             ins.get_xticklabels() + ins.get_yticklabels()):
    item.set_fontsize(16)

## Crop image

Skip this section if you do not require cropping. If you do require cropping, set `crop` to `True`.

In [ ]:
# PREVIEW CROP OPERATION
crop = False # change as needed
xmin,xmax = 450, 1650 # vertical cursor positions, change as needed
ymin,ymax = 50, 850 # horizontal cursor positions, change as needed

fig, ax = plt.subplots(1, 1, figsize=[12, 10])
arr = refstack_state_1[:,:,idx]/swpcount1
im = ax.imshow(arr,cmap='gray')
plt.plot([xmin, xmin],[0,imheight],color='crimson')
plt.plot([xmax, xmax],[0,imheight],color='crimson')
plt.plot([0, imwidth],[ymin,ymin],color='crimson')
plt.plot([0, imwidth],[ymax,ymax],color='crimson')
plt.xlim(0,imwidth)
plt.ylim(0,imheight)
ax.invert_yaxis()
ax.set_xlabel('x [pixels]')
ax.set_ylabel('y [pixels]');
for item in ([ax.title, ax.xaxis.label, ax.yaxis.label] +
             ax.get_xticklabels() + ax.get_yticklabels()):
    item.set_fontsize(20)

In [ ]:
# SHOW CROPPED IMAGE
plt.rcParams['font.size'] = 20
fig, ax = plt.subplots(1, 1, figsize=[12, 10])
ins = inset_axes(ax, width=3, height=1.5, loc='upper right', borderpad=2)
arr = refstack_state_1[ymin:ymax,xmin:xmax,idx]/swpcount1 
im = ax.imshow(arr,cmap='gray')
ax.set_xlabel('x [pixels]')
ax.set_ylabel('y [pixels]')
divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='5%', pad=.2)
cbar = fig.colorbar(im, cax=cax, orientation='vertical',format="%3.0f")
cbar.ax.tick_params(labelsize=16)
ins.hist(arr.ravel(),bins=np.arange(256),density=True,color='crimson')
ins.set_xlabel('Pixel value')
ins.tick_params(colors='crimson') 
ins.xaxis.label.set_color('crimson')
ins.set_facecolor((1,1,1,0))
ins.spines['left'].set_color((1,1,1,0))
ins.spines['right'].set_color((1,1,1,0))
ins.spines['top'].set_color((1,1,1,0))
ins.spines['bottom'].set_color('crimson')
ins.set_xticks([0,127,255])
ins.set_xlim(0,255)
ins.set_yticks([])
for item in ([ax.title, ax.xaxis.label, ax.yaxis.label] +
             ax.get_xticklabels() + ax.get_yticklabels()):
    item.set_fontsize(20)
for item in ([ins.title, ins.xaxis.label, ins.yaxis.label] +
             ins.get_xticklabels() + ins.get_yticklabels()):
    item.set_fontsize(16)

# DO CROP ON CONTRAST ARRAY
cdata_s1 = cdata_s1[ymin:ymax,xmin:xmax,:] if crop else cdata_s1
cdata_s2 = cdata_s2[ymin:ymax,xmin:xmax,:] if crop else cdata_s2

## Sort and divide data into sweep blocks

In [ ]:
# DO NOT MODIFY THIS CELL!

assert(numblock == 2)
rank = np.argsort(centerfreq).tolist()

freq1 = np.linspace(centerfreq[rank[0]]-span/2,centerfreq[rank[0]]+span/2,numfreq)
freq2 = np.linspace(centerfreq[rank[1]]-span/2,centerfreq[rank[1]]+span/2,numfreq)
cdata1_s1 = cdata_s1[:,:,rank[0]*numfreq:rank[0]*numfreq+numfreq]
cdata2_s1 = cdata_s1[:,:,rank[1]*numfreq:rank[1]*numfreq+numfreq]
cdata1_s2 = cdata_s2[:,:,rank[0]*numfreq:rank[0]*numfreq+numfreq]
cdata2_s2 = cdata_s2[:,:,rank[1]*numfreq:rank[1]*numfreq+numfreq]

## Plot global ODMR spectrum

In [ ]:
fig = plt.figure(figsize=(12,6))
plt.plot(freq1,np.mean(cdata1_s1,(0,1)),color='b',lw=2,label='State 1')
plt.plot(freq2,np.mean(cdata2_s1,(0,1)),color='b',lw=2,)
plt.plot(freq1,np.mean(cdata1_s2,(0,1)),color='r',lw=2,label='State 2')
plt.plot(freq2,np.mean(cdata2_s2,(0,1)),color='r',lw=2)
plt.xlim(freq1[0],freq2[-1])
plt.legend();

## Bin image

In [ ]:
hbin, vbin = 2,2 # bin size (px) in horizontal and vertical dimension, change as needed 

cdata1_s1_bin =  skim_blockred(cdata1_s1, (vbin, hbin, 1), np.average)[:-1,:-1]
cdata2_s1_bin =  skim_blockred(cdata2_s1, (vbin, hbin, 1), np.average)[:-1,:-1]
cdata1_s2_bin =  skim_blockred(cdata1_s2, (vbin, hbin, 1), np.average)[:-1,:-1]
cdata2_s2_bin =  skim_blockred(cdata2_s2, (vbin, hbin, 1), np.average)[:-1,:-1]

## Global Factor

In [ ]:
import numpy as np

def correct_global(data: np.ndarray, global_fraction: float) -> np.ndarray:
    """
    Python port of correct_global.m (QDMlab). Applies the global-illumination
    correction to a single-block ODMR contrast stack.
    """
    data = np.asarray(data, dtype=np.float64)

    spec_data = np.nanmean(data, axis=(0, 1))
    left_bl = spec_data[:5].mean()
    right_bl = spec_data[-6:].mean()
    global_mean_bl = (left_bl + right_bl) / 2.0

    baseline_range = global_mean_bl - spec_data.min()
    spec_zbl = spec_data - global_mean_bl

    left_bl_p = data[:, :, :5].mean(axis=2)
    right_bl_p = data[:, :, -6:].mean(axis=2)
    mean_baseline = (left_bl_p + right_bl_p) / 2.0
    pixel_range = mean_baseline - data.min(axis=2)

    with np.errstate(divide="ignore", invalid="ignore"):
        scale = baseline_range / pixel_range

    corrected = (
        scale[:, :, None] * (data - mean_baseline[:, :, None])
        - global_fraction * spec_zbl[None, None, :]
        + global_mean_bl
    )
    corrected[pixel_range == 0, :] = data[pixel_range == 0, :]
    return corrected

print("correct_global() defined.")

In [ ]:
import spectralFitFunctions as sff

# ============================================================
# CELL 1: Sweep global_fraction on RAW binned data (before any correction)
# ============================================================
REFERENCE_GF = 0.5  # for comparison

GF_SWEEP = np.array([0.0, 0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9, 1.0])
ROI_Y = slice(50, 100)
ROI_X = slice(400, 450)

roi_raw = cdata1_s1_bin[ROI_Y, ROI_X, :]  # must still be raw at this point
lwGuessMHz = 1.2

chi_vals = []
for gf in GF_SWEEP:
    roi_corrected = correct_global(roi_raw, gf)
    pG = sff.guessParams2111(roi_corrected.mean(axis=(0, 1)), freq1, lwGuessMHz=lwGuessMHz).astype('float32')
    pG_img = sff.guessParamsImg2111(roi_corrected, freq1, 5, 1, 5, 0.6, 5, pG[3], False)
    q_roi, chi_roi = sff.performLineshapeFitESR2111_GPU(roi_corrected, pG_img, freq1)
    chi_vals.append(np.nanmean(chi_roi))

chi_vals = np.array(chi_vals)
best_gf = GF_SWEEP[np.argmin(chi_vals)]
chi_best = chi_vals.min()
chi_worst = chi_vals.max()
chi_at_ref = np.interp(REFERENCE_GF, GF_SWEEP, chi_vals)

improvement_vs_ref = 100 * (chi_at_ref - chi_best) / chi_at_ref
improvement_vs_worst = 100 * (chi_worst - chi_best) / chi_worst

plt.figure(figsize=[8, 5])
plt.plot(GF_SWEEP, chi_vals, 'o-', color='#2b8a3e', lw=2, markersize=7)
plt.axvline(REFERENCE_GF, color='gray', ls=':', label=f'reference (gf={REFERENCE_GF})')
plt.axvline(best_gf, color='crimson', ls='--')
plt.scatter([best_gf], [chi_best], color='crimson', s=150, zorder=5, marker='*', label=f'optimal (gf={best_gf:.2f})')
plt.xlabel('global_fraction')
plt.ylabel('mean chi^2 (background ROI)')
plt.title(f'Optimal global_fraction = {best_gf:.2f}')
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print(f"Optimal global_fraction: {best_gf:.2f}")
print(f"Fit quality improves by {improvement_vs_ref:.1f}% at the optimum vs. reference (gf={REFERENCE_GF})")
print(f"Fit quality improves by {improvement_vs_worst:.1f}% at the optimum vs. the worst tested value")
print(f"\nLower chi^2 means more accurate fitted resonance frequencies (f1, f2),")
print(f"which directly improves Bz accuracy, reduces failed-fit pixels, and gives")
print(f"a cleaner background for downstream dipole detection and moment inversion.")
print(f"\n*** Use global_fraction = {best_gf:.2f} in the next cell ***")

In [ ]:
# ============================================================
# CELL 2: Apply the correction, keeping permanent raw copies
# ============================================================
APPLY_CORRECTION = True
global_fraction =  0.5 #best_gf or  from Cell 1 -- or set manually, e.g. 0.6

# permanent raw copies -- never overwritten again, safe to reference anytime
cdata1_s1_bin_raw = cdata1_s1_bin.copy()
cdata2_s1_bin_raw = cdata2_s1_bin.copy()
cdata1_s2_bin_raw = cdata1_s2_bin.copy()
cdata2_s2_bin_raw = cdata2_s2_bin.copy()

if APPLY_CORRECTION:
    cdata1_s1_bin = correct_global(cdata1_s1_bin_raw, global_fraction)
    cdata2_s1_bin = correct_global(cdata2_s1_bin_raw, global_fraction)
    cdata1_s2_bin = correct_global(cdata1_s2_bin_raw, global_fraction)
    cdata2_s2_bin = correct_global(cdata2_s2_bin_raw, global_fraction)
    print(f"Correction APPLIED (post-binning, global_fraction={global_fraction:.2f})")
else:
    print("Correction SKIPPED - cdata*_bin remain raw")

In [ ]:
# ============================================================
# CELL 3: Diagnostic plot -- always compares against the RAW copies
# ============================================================
data_raw = cdata1_s1_bin_raw          # guaranteed untouched
data_corrected = cdata1_s1_bin        # result from Cell 2

correction_magnitude = np.nanmean(np.abs(data_corrected - data_raw), axis=-1)

left_bl_p = data_raw[:, :, :5].mean(axis=2)
right_bl_p = data_raw[:, :, -6:].mean(axis=2)
mean_baseline = (left_bl_p + right_bl_p) / 2.0
pixel_range = mean_baseline - data_raw.min(axis=2)
uncorrected_fallback_mask = (pixel_range == 0)

n_fallback = np.sum(uncorrected_fallback_mask)
pct_fallback = 100 * n_fallback / uncorrected_fallback_mask.size
print(f"Pixels NOT corrected (zero dynamic range, fallback to raw): "
      f"{n_fallback} ({pct_fallback:.3f}% of binned image)")
print(f"Correction magnitude: mean={np.nanmean(correction_magnitude):.6f}  "
      f"max={np.nanmax(correction_magnitude):.6f}  "
      f"std={np.nanstd(correction_magnitude):.6f}")

ny, nx, nfreq = data_raw.shape
np.random.seed(0)
n_check = 6
check_pixels = [(np.random.randint(0, ny), np.random.randint(0, nx)) for _ in range(n_check)]
print(f"\nChecking binned pixels: {check_pixels}")

fig = plt.figure(figsize=[18, 12])

ax0 = plt.subplot(3, 3, 1)
im0 = ax0.imshow(correction_magnitude, cmap='inferno')
ax0.set_title('Per-pixel correction magnitude (binned)')
plt.colorbar(im0, ax=ax0, label='|Δcontrast|')
for y, x in check_pixels:
    ax0.plot(x, y, 'c+', markersize=12, markeredgewidth=2)

ax1 = plt.subplot(3, 3, 2)
ax1.imshow(uncorrected_fallback_mask, cmap='Reds')
ax1.set_title(f'Fallback (uncorrected) pixels\n{n_fallback} px ({pct_fallback:.3f}%)')

ax2 = plt.subplot(3, 3, 3)
mean_before = np.nanmean(data_raw, axis=(0, 1))
mean_after = np.nanmean(data_corrected, axis=(0, 1))
ax2.plot(mean_before, label='before')
ax2.plot(mean_after, label=f'after (gf={global_fraction:.2f})')
ax2.set_title('Whole-image mean spectrum (binned)')
ax2.legend()

for i, (y, x) in enumerate(check_pixels):
    ax = plt.subplot(3, 3, 4 + i)
    ax.plot(data_raw[y, x, :], 'b.-', label='before', markersize=4)
    ax.plot(data_corrected[y, x, :], 'r.-', label='after', markersize=4)
    ax.set_title(f'Binned pixel (y={y}, x={x})')
    ax.legend(fontsize=8)
    if i == 0:
        ax.set_ylabel('contrast')
    ax.set_xlabel('freq index')

plt.tight_layout()
plt.show()

## Generate guesses for fit

\[IMPORTANT!\] This code will only work if your data has only one doublet per sweep block. If this is not the case, do not proceed any further.

In [ ]:
use_global = False # True uses the global ODMR spectrum to inform guess generation for the image, False does not
lwGuessMHz = 1.2 # initial linewidth guess in MHz, change as needed
guessbinsize = 5 # how finely to further bin the image for guessing, increasing this number will speed up 
                    # execution but assumes slowly-varying shifts in the NV ODMR resonances 
nrow, ncol, nfr = cdata1_s1_bin.shape
npar = 5


if use_global:
    pG1_s1_global = sff.guessParams2111(cdata1_s1_bin.mean(axis=(0,1)), freq1, lwGuessMHz=lwGuessMHz).astype('float32')
    pG2_s1_global = sff.guessParams2111(cdata2_s1_bin.mean(axis=(0,1)), freq2, lwGuessMHz=lwGuessMHz).astype('float32')
    pG1_s2_global = sff.guessParams2111(cdata1_s2_bin.mean(axis=(0,1)), freq1, lwGuessMHz=lwGuessMHz).astype('float32')
    pG2_s2_global = sff.guessParams2111(cdata2_s2_bin.mean(axis=(0,1)), freq2, lwGuessMHz=lwGuessMHz).astype('float32')
    with Pool(2) as p:
        pG1_s1,pG2_s1 = p.starmap(sff.guessParamsImg2111, 
                                                [(cdata1_s1_bin, freq1, guessbinsize,1, 5, 0.6, 5, pG1_s1_global[3], False),
                                                 (cdata2_s1_bin, freq2, guessbinsize,1, 5, 0.6, 5, pG2_s1_global[3], False)])
        pG1_s2,pG2_s2 = p.starmap(sff.guessParamsImg2111, 
                                                [(cdata1_s2_bin, freq1, guessbinsize,1, 5, 0.6, 5, pG1_s2_global[3], False),
                                                 (cdata2_s2_bin, freq2, guessbinsize,1, 5, 0.6, 5, pG2_s2_global[3], False)])
else:
    with Pool(2) as p:
        pG1_s1,pG2_s1 = p.starmap(sff.guessParamsImg2111, 
                                                [(cdata1_s1_bin, freq1, guessbinsize,1, 5, 0.6, 5, lwGuessMHz, False),
                                                 (cdata2_s1_bin, freq2, guessbinsize,1, 5, 0.6, 5, lwGuessMHz, False)])
        pG1_s2,pG2_s2 = p.starmap(sff.guessParamsImg2111, 
                                                [(cdata1_s2_bin, freq1, guessbinsize,1, 5, 0.6, 5, lwGuessMHz, False),
                                                 (cdata2_s2_bin, freq2, guessbinsize,1, 5, 0.6, 5, lwGuessMHz, False)])

pG1_s1 = pG1_s1.astype('float32')
pG2_s1 = pG2_s1.astype('float32')
pG1_s2 = pG1_s2.astype('float32')
pG2_s2 = pG2_s2.astype('float32')

## Fit data on GPU

In [ ]:
q1_s1 = np.zeros((nrow, ncol, npar))
q1_s1, chi1_s1 = sff.performLineshapeFitESR2111_GPU(cdata1_s1_bin, pG1_s1, freq1)#, bSilent=True)
q2_s1 = np.zeros((nrow, ncol, npar))
q2_s1, chi2_s1 = sff.performLineshapeFitESR2111_GPU(cdata2_s1_bin, pG2_s1, freq2)#, bSilent=True)
f1_s1, f2_s1 = q1_s1[:,:,2], q2_s1[:,:,2]
g1_s1, g2_s1 = np.abs(q1_s1[:,:,3]), np.abs(q2_s1[:,:,3])

q1_s2 = np.zeros((nrow, ncol, npar))
q1_s2, chi1_s2 = sff.performLineshapeFitESR2111_GPU(cdata1_s2_bin, pG1_s2, freq1)#, bSilent=True)
q2_s2 = np.zeros((nrow, ncol, npar))
q2_s2, chi2_s2 = sff.performLineshapeFitESR2111_GPU(cdata2_s2_bin, pG2_s2, freq2)#, bSilent=True)
f1_s2, f2_s2 = q1_s2[:,:,2], q2_s2[:,:,2]
g1_s2, g2_s2 = np.abs(q1_s2[:,:,3]), np.abs(q2_s2[:,:,3])

## Visualize fitted parameters

The cell below plots the fitted center frequency ($f$), linewidth ($\Gamma$) and chi-squared ($\chi^2$) maps for each sweep block. 

In [ ]:
# FREQUENCIES
fig, ax = plt.subplots(1, 4, figsize=[8,8])
ax[0].set_xticks([]); ax[0].set_yticks([]); ax[0].set_title('State 1, $f_1$',fontsize=14); ax[0].imshow(f1_s1);
ax[1].set_xticks([]); ax[1].set_yticks([]); ax[1].set_title('State 1, $f_2$',fontsize=14); ax[1].imshow(f2_s1);
ax[2].set_xticks([]); ax[2].set_yticks([]); ax[2].set_title('State 2, $f_1$',fontsize=14); ax[2].imshow(f1_s2);
ax[3].set_xticks([]); ax[3].set_yticks([]); ax[3].set_title('State 2, $f_2$',fontsize=14); ax[3].imshow(f2_s2);
# LINEWIDTHS
fig, ax = plt.subplots(1, 4, figsize=[8,8])
ax[0].set_xticks([]); ax[0].set_yticks([]); ax[0].set_title('State 1, $\Gamma_1$',fontsize=14); ax[0].imshow(g1_s1);
ax[1].set_xticks([]); ax[1].set_yticks([]); ax[1].set_title('State 1, $\Gamma_2$',fontsize=14); ax[1].imshow(g2_s1);
ax[2].set_xticks([]); ax[2].set_yticks([]); ax[2].set_title('State 2, $\Gamma_1$',fontsize=14); ax[2].imshow(g1_s1);
ax[3].set_xticks([]); ax[3].set_yticks([]); ax[3].set_title('State 2, $\Gamma_2$',fontsize=14); ax[3].imshow(g2_s1);
# CHI-SQUARED
fig, ax = plt.subplots(1, 4, figsize=[8,8])
ax[0].set_xticks([]); ax[0].set_yticks([]); ax[0].set_title('State 1, $\chi^2_1$',fontsize=14); ax[0].imshow(chi1_s1);
ax[1].set_xticks([]); ax[1].set_yticks([]); ax[1].set_title('State 1, $\chi^2_2$',fontsize=14); ax[1].imshow(chi2_s1);
ax[2].set_xticks([]); ax[2].set_yticks([]); ax[2].set_title('State 2, $\chi^2_1$',fontsize=14); ax[2].imshow(chi1_s2);
ax[3].set_xticks([]); ax[3].set_yticks([]); ax[3].set_title('State 2, $\chi^2_2$',fontsize=14); ax[3].imshow(chi1_s2);

## Compute magnetic fields and crystal strain

\[IMPORTANT!\] If you only have one sweep block, do not proceed any further. $B_{z,k}$ (in units of µT) is the magnetic field projection along the quantization axis ($z$ in NV coodinate system) for the subset of NVs in orientation $k$ (ordered from maximal to minimal projection if there are more than 2 sweep blocks).

Similarly, $M_{z,k}$ (in units of kHz) are the crystal strain components along each orientation $k$.

In [ ]:
# DO NOT MODIFY THIS CELL!

Bz_s1 = (q2_s1[:,:,2] - q1_s1[:,:,2])/5.6*100
Mz_s1 = (q2_s1[:,:,2] + q1_s1[:,:,2])/2*1000
Bz_s2 = (q2_s2[:,:,2] - q1_s2[:,:,2])/5.6*100
Mz_s2 = (q2_s2[:,:,2] + q1_s2[:,:,2])/2*1000

# BACKGROUND SUBTRACTION

bg_subtract = True # change as needed, typ. True
rm_outliers, thresh = True, 3  # change as needed, typ. True
order = 1 # 1 for plane (default), 2 for bivariate quadratic, change as needed

def remove_outliers(A,thresh):
    out = np.zeros_like(A)
    med = np.median(A)
    MAD = median_abs_deviation(A,axis=None)
    scaled_MAD = MAD/0.67449
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            if np.abs(A[i,j]-med) > thresh*scaled_MAD:
                out[i,j] = med
            else:
                out[i,j] = A[i,j]
    return out
    
if bg_subtract:
    if rm_outliers:
        Bz_s1 -= sff.computeBG2D(remove_outliers(Bz_s1,thresh), order)
        Mz_s1 -= np.median(remove_outliers(Mz_s1,thresh))
        Bz_s2 -= sff.computeBG2D(remove_outliers(Bz_s2,thresh), order)
        Mz_s2 -= np.median(remove_outliers(Mz_s2,thresh))
    else:
        Bz_s1 -= sff.computeBG2D(Bz_s1, order)
        Mz_s1 -= np.median(Mz_s1)
        Bz_s2 -= sff.computeBG2D(Bz_s2, order)
        Mz_s2 -= np.median(Mz_s2)

In [ ]:
Bmin, Bmax = -20,20 # colorbar limits for magnetic field in uT, change as needed
save_pdf = True # change as needed
plt.rcParams['font.size'] = 16
interp_method = 'none' # change as needed, 'none' (default) or 'gaussian' for smoothed image

fig, ax = plt.subplots(2, 2, figsize=[16,12])
im = ax[0,0].imshow(Bz_s1,cmap='RdBu_r',vmin=Bmin,vmax=Bmax, interpolation=interp_method)
ax[0,0].set_xlabel('x [binned pixels]')
ax[0,0].set_ylabel('y [binned pixels]')
ax[0,0].set_title('State 1 $B_{111}$')
divider = make_axes_locatable(ax[0,0])
cax = divider.append_axes('right', size='5%', pad=.2)
cbar = fig.colorbar(im, cax=cax, orientation='vertical',format="%3.0f");
cbar.ax.tick_params(labelsize=14)
cbar.ax.set_title('µT',fontsize=14)

im = ax[0,1].imshow(Bz_s2,cmap='RdBu_r',vmin=Bmin,vmax=Bmax, interpolation=interp_method)
ax[0,1].set_xlabel('x [binned pixels]')
ax[0,1].set_ylabel('y [binned pixels]')
ax[0,1].set_title('State 2 $B_{111}$')
divider = make_axes_locatable(ax[0,1])
cax = divider.append_axes('right', size='5%', pad=.2)
cbar = fig.colorbar(im, cax=cax, orientation='vertical',format="%3.0f");
cbar.ax.tick_params(labelsize=14)
cbar.ax.set_title('µT',fontsize=14)

im = ax[1,0].imshow((Bz_s1+Bz_s2)/2,cmap='RdBu_r',vmin=Bmin,vmax=Bmax, interpolation=interp_method)
ax[1,0].set_xlabel('x [binned pixels]')
ax[1,0].set_ylabel('y [binned pixels]')
ax[1,0].set_title('(State 1 $B_{111}$ + State 2 $B_{111}$)/2')
divider = make_axes_locatable(ax[1,0])
cax = divider.append_axes('right', size='5%', pad=.2)
cbar = fig.colorbar(im, cax=cax, orientation='vertical',format="%3.0f");
cbar.ax.tick_params(labelsize=14)
cbar.ax.set_title('µT',fontsize=14)

im = ax[1,1].imshow((Bz_s1-Bz_s2)/2,cmap='RdBu_r',vmin=Bmin,vmax=Bmax, interpolation=interp_method)
ax[1,1].set_xlabel('x [binned pixels]')
ax[1,1].set_ylabel('y [binned pixels]')
ax[1,1].set_title('(State 1 $B_{111}$ - State 2 $B_{111}$)/2')
divider = make_axes_locatable(ax[1,1])
cax = divider.append_axes('right', size='5%', pad=.2)
cbar = fig.colorbar(im, cax=cax, orientation='vertical',format="%3.0f");
cbar.ax.tick_params(labelsize=14)
cbar.ax.set_title('µT',fontsize=14)

fig.tight_layout(h_pad=-1, w_pad=2)

if save_pdf:
    plt.savefig('B111_map.pdf', format='pdf', dpi=600, bbox_inches='tight')

For more advanced troubleshooting, you can select any of the maps above using `arraysource` to examine more closely, along with the corresponding ODMR spectrum and fit overlaid at the crosshair location.

In [ ]:
arraysource = Bz_s1

freqsource1 = freq1
parsource1 = q1_s1
datasource1 = cdata1_s1_bin
freqsource2 = freq2
parsource2 = q2_s1
datasource2 = cdata2_s1_bin

imwidth_bin = arraysource.shape[1]
imheight_bin = arraysource.shape[0]
asp_ratio = imwidth_bin/imheight_bin

plot1 = figure(height=200, width=300)
plot1.x_range.range_padding = plot1.y_range.range_padding = 0
plot1.xgrid.visible, plot1.ygrid.visible = False, False
data1 = plot1.line(freqsource1, datasource1[0,0,:], color='blue')
fit1 = plot1.line(freqsource1, sff.specESR2111(parsource1[0,0,:], freqsource1), color='red')
plot1.yaxis.minor_tick_line_color = None
plot1.yaxis.minor_tick_line_color = None

plot2 = figure(height=200, width=300)
plot2.x_range.range_padding = plot2.y_range.range_padding = 0
plot2.xgrid.visible, plot2.ygrid.visible = False, False
data2 = plot2.line(freqsource2, datasource2[0,0,:], color='blue')
fit2 = plot2.line(freqsource2, sff.specESR2111(parsource2[0,0,:], freqsource2), color='red')
plot2.yaxis.minor_tick_line_color = None

plot3 = figure(tooltips=[("x", "$x"), ("y", "$y"), ("value", "@image")], height=int(600/asp_ratio), width=600)
plot3.x_range.range_padding = plot3.y_range.range_padding = 0
plot3.axis.visible = False
cmin,cmax = Bmin,Bmax
image_glyph = plot3.image(image=[np.clip(arraysource,cmin,cmax)], x=0, y=0, dw=imwidth_bin, dh=imheight_bin, palette="Viridis256")
plot3.y_range = Range1d(start=imheight_bin, end=0)
vcursor = plot3.line([0,0], [0,imheight_bin], color="#440154", line_width=2)
hcursor = plot3.line([0,imwidth_bin], [0,0], color="#440154", line_width=2)

def update(x,y):
    vcursor.data_source.data['x'] = [x,x]
    hcursor.data_source.data['y'] = [y,y]
    data1.data_source.data['y'] = datasource1[y,x,:]
    fit1.data_source.data['y'] = sff.specESR2111(parsource1[y,x,:],freqsource1)
    data2.data_source.data['y'] = datasource2[y,x,:]
    fit2.data_source.data['y'] = sff.specESR2111(parsource2[y,x,:],freqsource2)
    push_notebook()
    
# Dropdown selection handler
def select_state(state):
    global arraysource, freqsource1, parsource1, datasource1, freqsource2, parsource2, datasource2, imwidth_bin, imheight_bin, asp_ratio
    
    # Get current slider values
    current_x = slider_x.value
    current_y = slider_y.value
    
    if state == 'State 1':
        arraysource = Bz_s1
        freqsource1 = freq1
        parsource1 = q1_s1
        datasource1 = cdata1_s1_bin
        freqsource2 = freq2
        parsource2 = q2_s1
        datasource2 = cdata2_s1_bin
    else:
        arraysource = Bz_s2
        freqsource1 = freq1
        parsource1 = q1_s2
        datasource1 = cdata1_s2_bin
        freqsource2 = freq2
        parsource2 = q2_s2
        datasource2 = cdata2_s2_bin

    # Update image dimensions and aspect ratio
    imwidth_bin = arraysource.shape[1]
    imheight_bin = arraysource.shape[0]
    asp_ratio = imwidth_bin / imheight_bin
    
    # Refresh plot data
    image_glyph.data_source.data['image'] = [np.clip(arraysource, cmin, cmax)]
    
    # Update plot1 and plot2 data
    data1.data_source.data['x'] = freqsource1
    data1.data_source.data['y'] = datasource1[current_y,current_x,:]  # Reset to some default or initial values
    fit1.data_source.data['x'] = freqsource1
    fit1.data_source.data['y'] = sff.specESR2111(parsource1[current_y,current_x,:],freqsource1)  # Replace with actual fit function

    data2.data_source.data['x'] = freqsource2
    data2.data_source.data['y'] = datasource2[current_y,current_x,:]  # Reset to some default or initial values
    fit2.data_source.data['x'] = freqsource2
    fit2.data_source.data['y'] = sff.specESR2111(parsource2[current_y,current_x,:],freqsource2)  # Replace with actual fit function
    
    push_notebook()

# Dropdown widget
dropdown = Dropdown(
    options=['State 1', 'State 2'],
    value='State 1',
    description='Select State:',
)

# Sliders for x and y positions
slider_x = IntSlider(min=0, max=imwidth_bin - 1, value=0, description='X:')
slider_y = IntSlider(min=0, max=imheight_bin - 1, value=0, description='Y:')

# Set dropdown callback
dropdown.observe(lambda change: select_state(change.new), names='value')

# Show the interactive plot and widgets
show(column(row(plot1, plot2), plot3), notebook_handle=True)
interact(update, x=slider_x, y=slider_y);
display(dropdown)

## Re-fit data on GPU

If your fits failed to converge for some reason, you can use automatic re-fitting below to take results from converged pixels to inform initial guesses for unconverged pixels.

In [ ]:
fErrThreshMHz = 6.0 # error threshold for center frequency, change as needed (typ. half of MW sweep span)
aErrThresh = 0.05 # error threshold for peak amplitudes, change as needed (typ. 0.05)
filt_sigma = 1.0 # scale (px) of influence (typ. 1.0)
nIter = 10 # number of refitting iterations


q1_s1, chi1_s1 = sff.lineshapeRefit_GPU(cdata1_s1_bin, freq1, q1_s1, fErrThreshMHz, aErrThresh, filt_sigma, nIter)
q2_s1, chi2_s1 = sff.lineshapeRefit_GPU(cdata2_s1_bin, freq2, q2_s1, fErrThreshMHz, aErrThresh, filt_sigma, nIter)
f1_s1, f2_s1 = q1_s1[:,:,2], q2_s1[:,:,2]
g1_s1, g2_s1 = np.abs(q1_s1[:,:,3]), np.abs(q2_s1[:,:,3])
q1_s2, chi1_s2 = sff.lineshapeRefit_GPU(cdata1_s2_bin, freq1, q1_s2, fErrThreshMHz, aErrThresh, filt_sigma, nIter)
q2_s2, chi2_s2 = sff.lineshapeRefit_GPU(cdata2_s2_bin, freq2, q2_s2, fErrThreshMHz, aErrThresh, filt_sigma, nIter)
f1_s2, f2_s2 = q1_s2[:,:,2], q2_s2[:,:,2]
g1_s2, g2_s2 = np.abs(q1_s2[:,:,3]), np.abs(q2_s2[:,:,3])

At this point, you may want to revisit the cells above (starting from the "Visualize fitted parameters" section) to see if re-fitting has helped.

## Dump output to file

In [ ]:
save_mat = True # change as needed

if save_mat:
    savemat("B111_s1.mat", {"B111_s1": Bz_s1})
    savemat("B111_s2.mat", {"B111_s2": Bz_s2})
    savemat("B111ferro.mat", {"B111ferro": (Bz_s1+Bz_s2)/2})
    savemat("B111para.mat", {"B111para": (Bz_s1-Bz_s2)/2})